In [1]:
# Filename......: L12_OlisBahari_ITAI1371
# Language......: Python
# Tools.........: Visual Studio Code (VSC)
#               : Google Colab
# Class.........: ITAI 1371 Introduction to Machine Learning
# Semester......: Summer 2026
# Class Type....: Online
# Instructor....: Sitaram Ayyagari
# Student.......: Olis Bahari
# Version.......: V1.0
# Purpose.......: Train a logistic regression model and audit its
#                 fairness across male and female demographic groups.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix

In [3]:
# Load the data
url = (
    "https://archive.ics.uci.edu/ml/"
    "machine-learning-databases/adult/adult.data"
)

columns = [
    "age",
    "workclass",
    "fnlwgt",
    "education",
    "education-num",
    "marital-status",
    "occupation",
    "relationship",
    "race",
    "sex",
    "capital-gain",
    "capital-loss",
    "hours-per-week",
    "native-country",
    "income"
]

df = pd.read_csv(
    url,
    header=None,
    names=columns,
    sep=r",\s*",
    engine="python",
    na_values="?"
)

# Data Cleaning

# Remove rows containing missing values
df.dropna(inplace=True)

# Convert income labels into binary values
# <=50K = 0
# >50K  = 1
df['income'] = df['income'].map({'<=50K': 0, '>50K': 1})

# Separate features and target
X = df.drop('income', axis=1)
y = df['income']

# Split the Dataset Into Train and Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Create a preprocessing pipeline
numeric_features = X.select_dtypes(include='number').columns
categorical_features = X.select_dtypes(exclude='number').columns

preprocessor = make_column_transformer(
    (StandardScaler(), numeric_features),
    (OneHotEncoder(handle_unknown='ignore'), categorical_features)
)

# Train a baseline model using logistic regression
model = make_pipeline(preprocessor, LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)

print(f"Overall model accuracy: {model.score(X_test, y_test):.2%}")

Overall model accuracy: 84.61%


In [4]:
def get_subgroup_accuracy(model, X_test, y_test, subgroup_column, subgroup_value):
    """Calculates accuracy for a specific subgroup of the test data."""
    # Create a boolean mask to select the subgroup from X_test
    subgroup_mask = X_test[subgroup_column] == subgroup_value
    
    # Select the subgroup data
    X_subgroup = X_test[subgroup_mask]
    y_subgroup = y_test[subgroup_mask]
    
    # Calculate and return the model's score on this subgroup
    return model.score(X_subgroup, y_subgroup)

# Calculate accuracy for males and females
acc_male = get_subgroup_accuracy(model, X_test, y_test, 'sex', 'Male')
acc_female = get_subgroup_accuracy(model, X_test, y_test, 'sex', 'Female')

print(f"Accuracy for Males: {acc_male:.2%}")
print(f"Accuracy for Females: {acc_female:.2%}")

Accuracy for Males: 81.20%
Accuracy for Females: 91.81%


In [5]:
# False Positive and False Negative Rates
def get_rates(model, X_test, y_test, subgroup_column, subgroup_value):

    subgroup_mask = X_test[subgroup_column] == subgroup_value
    X_subgroup = X_test[subgroup_mask]
    y_subgroup = y_test[subgroup_mask]
    
    # Generate predictions
    y_pred_subgroup = model.predict(X_subgroup)
    tn, fp, fn, tp = confusion_matrix(y_subgroup, y_pred_subgroup).ravel()
    
    # Calculate rates safely
    fpr = fp / (fp + tn)
    fnr = fn / (fn + tp)
    return fpr, fnr


# Calculate the rates for males
fpr_male, fnr_male = get_rates(model, X_test, y_test, 'sex', 'Male')

# Calculate the rates for females
fpr_female, fnr_female = get_rates(model, X_test, y_test, 'sex', 'Female')

print(f"Male - False Positive Rate: {fpr_male:.2%}, False Negative Rate: {fnr_male:.2%}")
print(f"Female - False Positive Rate: {fpr_female:.2%}, False Negative Rate: {fnr_female:.2%}")

Male - False Positive Rate: 10.26%, False Negative Rate: 37.80%
Female - False Positive Rate: 2.81%, False Negative Rate: 47.84%


Reflective Knowledge Check

1. Analyze Your Results

Accuracy for Males: 81.20%

Accuracy for Females: 91.81%

The model demonstrates a 10.61 percentage point difference in accuracy, performing at 81.20% for males and 91.81% for females. It is more accurate for females overall.

Reflective Knowledge Check

2. Interpret the Errors

The False Positive Rate (FPR) was 10.26% for males and 2.81% for females. This means the model is more likely to make false positive errors for males, so male applicants without high incomes are more often misclassified as having high incomes.

In loan applications, this error can lead lenders to approve loans or offer better terms to people whose real income does not support the loan. This raises financial risk for lenders and may put applicants at risk of debt they cannot manage.

The False Negative Rate (FNR) shows another fairness issue. The FNR was 37.80% for males and 47.84% for females, which means high-income females are more likely than males to be misclassified as having lower incomes.

Reflective Knowledge Check

3. Justify a Decision

I would not approve this model for screening candidates for high-paying jobs, as a false negative could result in a qualified individual being unfairly rejected and losing a valuable opportunity.

The model's false negative rate is 47.84% for females and 37.80% for males, a difference of nearly 10 percentage points. This indicates the model is more likely to overlook qualified high-income female candidates. Although overall accuracy is higher for females, the elevated false negative rate increases the risk of rejecting qualified female applicants. The false positive rates also differ, at 10.26% for males and 2.81% for females, indicating errors are not distributed evenly. I recommend conducting additional fairness assessments and applying bias reduction techniques before approving the model.

Reflective Knowledge Check

4. Propose a Mitigation

Removing the sex column does not ensure fairness, as other variables may still correlate with sex and convey similar information. For example, occupation, relationship status, marital status, education, and hours worked per week often differ between men and women.

The model may still detect patterns related to sex, even without the sex column. While removing a sensitive attribute can reduce bias, it is important to reassess fairness after retraining. Recalculate subgroup accuracy, false positive rate (FPR), and false negative rate (FNR) to determine whether differences between men and women have decreased.